# trt_yolo_bench - Colab Runner

This notebook runs the repo pipeline in Colab from clone to scoring.
Run cells top-to-bottom.

## 1) Runtime checks
In Colab: Runtime -> Change runtime type -> GPU (T4 recommended).

In [1]:
!nvidia-smi
!nvcc --version

Sun Jul 12 04:56:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Clone or enter repo

In [9]:
import os
from pathlib import Path

REPO_URL = "https://github.com/chao-dotcom/trt_yolo_bench.git"
REPO_DIR = Path("/content/trt_yolo_bench")

if not REPO_DIR.exists():
    !git clone {REPO_URL} /content/trt_yolo_bench
else:
    %cd /content/trt_yolo_bench
    !git pull --ff-only

%cd /content/trt_yolo_bench
!git status --short

/content/trt_yolo_bench
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.05 KiB | 1.05 MiB/s, done.
From https://github.com/chao-dotcom/trt_yolo_bench
   a3826c5..3ac9282  main       -> origin/main
Updating a3826c5..3ac9282
Fast-forward
 harness/main.cpp | 60 +++++++++++++++++++++++++++++---------------------------
 1 file changed, 31 insertions(+), 29 deletions(-)
/content/trt_yolo_bench
?? data/calib_slice.json
?? data/train2017/
?? data/val2017/
?? data/val_slice.json
?? yolo11n.pt


## 3) Install Python dependencies

In [3]:
!python -m pip install -q --upgrade pip
!pip install -q ultralytics onnx pycocotools requests opencv-python numpy

# TensorRT / CUDA deps.
# Pinned to the last TensorRT 10.x release: TensorRT 11.0 removed the implicit
# INT8 calibrator API (IInt8EntropyCalibrator2) that calibrator.py depends on.
!pip install -q tensorrt-cu12==10.16.1.11 || pip install -q tensorrt==10.16.1.11
!pip install -q pycuda

# OpenCV C++ dev headers/cmake config for the harness build (opencv-python above
# only ships its own bundled runtime libs, not the dev package find_package needs).
!apt-get -qq update && apt-get -qq install -y libopencv-dev

# Fail fast here instead of deep inside calibrator.py at step 6 if a future
# Colab image silently resolves a newer/incompatible TensorRT build.
import tensorrt as trt

assert hasattr(trt, "IInt8EntropyCalibrator2"), (
    f"tensorrt {trt.__version__} is missing IInt8EntropyCalibrator2 "
    "(removed in TensorRT 11+). Re-pin the install above to a 10.x release."
)
print(f"[ok] tensorrt {trt.__version__} has IInt8EntropyCalibrator2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxt

## 4) Prepare deterministic COCO slices + images

In [4]:
!python export/download_coco_slice.py

[download] http://images.cocodataset.org/annotations/annotations_trainval2017.zip -> data/annotations_trainval2017.zip
[extract] data/annotations_trainval2017.zip
[write] data/instances_val2017.json
[write] data/instances_train2017.json
[write] data/val_slice.json (1000 entries)
[write] data/calib_slice.json (200 entries)
[download] http://images.cocodataset.org/val2017/000000000139.jpg -> data/val2017/000000000139.jpg
[download] http://images.cocodataset.org/val2017/000000000285.jpg -> data/val2017/000000000285.jpg
[download] http://images.cocodataset.org/val2017/000000000632.jpg -> data/val2017/000000000632.jpg
[download] http://images.cocodataset.org/val2017/000000000724.jpg -> data/val2017/000000000724.jpg
[download] http://images.cocodataset.org/val2017/000000000776.jpg -> data/val2017/000000000776.jpg
[download] http://images.cocodataset.org/val2017/000000000785.jpg -> data/val2017/000000000785.jpg
[download] http://images.cocodataset.org/val2017/000000000802.jpg -> data/val2017/

## 5) Export ONNX

In [5]:
!python export/export_onnx.py --weights yolo11n.pt --out-dir models

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from 'yolo11n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (5.4 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 361ms
Prepared 3 packages in 339ms
Installed 3 packages in 8ms
 + colorama==0.4.6


## 6) Build TensorRT engines (FP32/FP16/INT8)

In [6]:
!python export/build_engines.py --onnx models/yolo11n.onnx --engines-dir engines

[build] precision=fp32
[07/12/2026-05:27:12] [TRT] [I] [MemUsageChange] Init CUDA: CPU +1, GPU +0, now: CPU 38, GPU 232 (MiB)
[07/12/2026-05:27:12] [TRT] [I] ----------------------------------------------------------------
[07/12/2026-05:27:12] [TRT] [I] ONNX IR version:  0.0.8
[07/12/2026-05:27:12] [TRT] [I] Opset version:    17
[07/12/2026-05:27:12] [TRT] [I] Producer name:    pytorch
[07/12/2026-05:27:12] [TRT] [I] Producer version: 2.11.0
[07/12/2026-05:27:12] [TRT] [I] Domain:           
[07/12/2026-05:27:12] [TRT] [I] Model version:    0
[07/12/2026-05:27:12] [TRT] [I] Doc string:       
[07/12/2026-05:27:12] [TRT] [I] ----------------------------------------------------------------
[07/12/2026-05:27:13] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU +281, GPU +4, now: CPU 533, GPU 236 (MiB)
[07/12/2026-05:27:13] [TRT] [I] Local timing cache in use. Profiling results in this builder pass will not be stored.
[07/12/2026-05:27:58] [TRT] [I] Compiler backend is used dur

## 7) Build C++ harness

In [10]:
import glob
import os
import subprocess
import sysconfig

os.chdir("/content/trt_yolo_bench/harness")

# The pip-installed tensorrt-cu12 wheel ships the runtime .so files but not the
# C++ headers (NvInfer.h etc). Fetch just the matching headers from NVIDIA's OSS
# repo (v10.16 matches the pinned tensorrt-cu12==10.16.1.11 from step 3) so the
# engine-building step and this harness link against the identical TensorRT build.
if not os.path.isdir("/content/tensorrt_oss/include"):
    subprocess.run(
        [
            "git", "clone", "--branch", "v10.16", "--depth", "1",
            "--filter=blob:none", "--sparse",
            "https://github.com/NVIDIA/TensorRT.git", "/content/tensorrt_oss",
        ],
        check=True,
    )
    subprocess.run(
        ["git", "-C", "/content/tensorrt_oss", "sparse-checkout", "set", "include"],
        check=True,
    )


def find_lib(pattern: str) -> str:
    site = sysconfig.get_paths()["purelib"]
    matches = sorted(glob.glob(os.path.join(site, "**", pattern), recursive=True))
    if not matches:
        raise FileNotFoundError(f"{pattern} not found under {site}")
    return matches[-1]


trt_lib = find_lib("libnvinfer.so*")
trt_lib_dir = os.path.dirname(trt_lib)
print(f"TensorRT include: /content/tensorrt_oss/include")
print(f"TensorRT lib: {trt_lib}")

subprocess.run(
    [
        "cmake", "-S", ".", "-B", "build",
        "-DTENSORRT_INCLUDE_DIR=/content/tensorrt_oss/include",
        f"-DTENSORRT_LIB={trt_lib}",
        f"-DCMAKE_EXE_LINKER_FLAGS=-Wl,-rpath,{trt_lib_dir}",
    ],
    check=True,
)
subprocess.run(["cmake", "--build", "build", "-j"], check=True)

os.chdir("/content/trt_yolo_bench")

TensorRT include: /content/tensorrt_oss/include
TensorRT lib: /usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer.so.10


## 8) Run harness (start with FP16)

In [11]:
!./harness/build/trt_yolo_bench engines/yolo11n_fp16.engine data/val_slice.json --out results_fp16.json

[info] input dims: 1 3 640 640
[info] output dims: 1 84 8400
[bench] mean_ms=1.422 p50_ms=1.463 p99_ms=2.003 fps=703.029
[write] results_fp16.json detections=5024


## 9) Score COCO metrics

In [12]:
!python eval/score_coco.py --dets results_fp16.json

loading annotations into memory...
Done (t=0.56s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.49s).
Accumulating evaluation results...
DONE (t=0.62s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.351
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.467
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.385
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.115
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.392
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.530
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.286
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.394
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

## 10) Optional: run FP32 and INT8
Uncomment and run if engine files were built successfully.

In [14]:
!./harness/build/trt_yolo_bench engines/yolo11n_fp32.engine data/val_slice.json --out results_fp32.json
!python eval/score_coco.py --dets results_fp32.json
!./harness/build/trt_yolo_bench engines/yolo11n_int8.engine data/val_slice.json --out results_int8.json
!python eval/score_coco.py --dets results_int8.json

[info] input dims: 1 3 640 640
[info] output dims: 1 84 8400
[bench] mean_ms=1.311 p50_ms=1.310 p99_ms=1.318 fps=762.995
[write] results_fp32.json detections=5020
loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.01s).
Accumulating evaluation results...
DONE (t=0.71s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.350
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.467
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.384
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.115
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.389
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.529
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxD